# L4 Residual Clustering

Clusters the low-confidence listings from `l4_taxonomy_classification.ipynb` to identify:
1. **Missing L4 categories** — clusters that don't map cleanly to any existing anchor
2. **Anchor description gaps** — correctly-assigned items scoring low due to thin anchor wording

No new embeddings are generated — all vectors are retrieved from the existing cache.

**Input:** `artifacts/analysis/l4_classification_validate_products/residual_for_clustering.csv`  
**Output:** `artifacts/analysis/l4_residual_clustering/cluster_results.csv` + sample printouts

In [ ]:
import sys
from pathlib import Path

# Resolve project root (two levels up from notebooks/)
PROJECT_ROOT = Path("__file__").resolve().parent.parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pickle
import numpy as np
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt

from product_classifier_utils import (
    load_product_data,
    build_product_text,
    stable_text_hash,
    get_bedrock_client,
)

print("Imports OK")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

# Input: residual from the classification run
RESIDUAL_CSV = PROJECT_ROOT / "artifacts/analysis/l4_classification_validate_products/residual_for_clustering.csv"

# Source table to reload product names/text for display and hashing
PRODUCTS_TABLE = "SNOWFLAKE_LEARNING_DB.SMCMAHON_PRODUCTS.PRODUCTS_LEI"
AWS_PROFILE    = "staging.admin"

# Embedding cache — no new Bedrock calls will be made
CACHE_PATH = PROJECT_ROOT / "artifacts/cache/embedding_cache.pkl"

# Clustering
N_CLUSTERS      = 30    # Start here; adjust after reviewing elbow plot
RANDOM_STATE    = 42
SAMPLE_PER_CLUSTER = 15  # Listings to print per cluster for manual review

# Optional: cluster separately within each assigned-L4 group
# True  → smaller, more coherent clusters (good for L5 discovery)
# False → full residual together (good for catching missing L4s)
CLUSTER_PER_L4 = False

# LLM cluster labeling (optional — set True only if Claude access confirmed)
RUN_LLM_LABELING = False
LLM_MODEL_ID     = "us.anthropic.claude-3-haiku-20241022-v1:0"
LLM_MAX_CLUSTERS = 30
LLM_TEMPERATURE  = 0.2

# Output
OUTPUT_DIR = PROJECT_ROOT / "artifacts/analysis/l4_residual_clustering"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Residual CSV : {RESIDUAL_CSV}")
print(f"Cache        : {CACHE_PATH}")
print(f"N_CLUSTERS   : {N_CLUSTERS}")
print(f"Per-L4 mode  : {CLUSTER_PER_L4}")
print(f"Output dir   : {OUTPUT_DIR}")

## 1. Load Residual IDs

In [ ]:
residual_meta = pd.read_csv(RESIDUAL_CSV)
residual_ids  = set(residual_meta["PRODUCT_ID"].astype(str))

print(f"Residual listings : {len(residual_ids):,}")
print(f"Columns in CSV    : {list(residual_meta.columns)}")
print()
print("L4 breakdown of residual:")
print(residual_meta["ASSIGNED_L4_LABEL"].value_counts().to_string())

## 2. Load Product Data from Snowflake (for text + display names)

We reload from Snowflake to get product names and descriptions needed to:
- Build the text strings for cache lookups
- Display human-readable samples per cluster

No embeddings are generated — everything comes from the cache.

In [ ]:
print(f"Loading product data from {PRODUCTS_TABLE} ...")
raw_df = load_product_data(
    table=PRODUCTS_TABLE,
    aws_profile=AWS_PROFILE,
    row_limit=None,
)
print(f"Loaded {len(raw_df):,} total rows")

# Filter to residual only
raw_df["PRODUCT_ID"] = raw_df["PRODUCT_ID"].astype(str)
df = raw_df[raw_df["PRODUCT_ID"].isin(residual_ids)].copy()
print(f"Filtered to residual: {len(df):,} rows")

# Build product text (same logic used during classification)
df["PRODUCT_TEXT"] = build_product_text(df)
df["TEXT_HASH"]    = df["PRODUCT_TEXT"].apply(stable_text_hash)

# Merge in assigned L4 label from residual CSV
df = df.merge(
    residual_meta[["PRODUCT_ID", "ASSIGNED_L4_LABEL", "ASSIGNED_L4_ID", "CONFIDENCE"]],
    on="PRODUCT_ID",
    how="left",
)

print(f"Final working set: {len(df):,} rows")

## 3. Retrieve Embeddings from Cache

No Bedrock calls — vectors are pulled directly from the local cache.

In [ ]:
print(f"Loading embedding cache from {CACHE_PATH} ...")
with open(CACHE_PATH, "rb") as f:
    cache = pickle.load(f)
print(f"Cache entries: {len(cache):,}")

# Retrieve vectors for residual items
vectors, valid_mask = [], []
for h in df["TEXT_HASH"]:
    if h in cache:
        vectors.append(cache[h])
        valid_mask.append(True)
    else:
        valid_mask.append(False)

valid_mask = np.array(valid_mask)
df_valid   = df[valid_mask].copy().reset_index(drop=True)
X          = normalize(np.array(vectors), norm="l2")  # unit-normalize for cosine distance

missing = (~valid_mask).sum()
print(f"Cache hits   : {valid_mask.sum():,}")
print(f"Cache misses : {missing:,}  ← should be 0; if not, re-run classification with RUN_EMBEDDING=True")
print(f"Embedding dim: {X.shape[1]}")

## 4. Elbow Plot — Choose K

In [ ]:
k_range  = range(10, 51, 5)
inertias = []

print("Running elbow sweep (MiniBatchKMeans)...")
for k in k_range:
    km = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=3, batch_size=4096)
    km.fit(X)
    inertias.append(km.inertia_)
    print(f"  k={k:3d}  inertia={km.inertia_:,.0f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(k_range), inertias, marker="o", color="steelblue")
ax.axvline(N_CLUSTERS, color="tomato", linestyle="--", label=f"Selected K={N_CLUSTERS}")
ax.set_xlabel("K")
ax.set_ylabel("Inertia")
ax.set_title("Elbow Plot — Residual Clustering")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nReview the elbow above, then update N_CLUSTERS in the config cell and re-run from Cell 5 onward.")

## 5. Cluster

In [ ]:
if CLUSTER_PER_L4:
    # Cluster within each assigned-L4 group independently
    print("Clustering per assigned L4 ...")
    df_valid["CLUSTER"] = -1
    cluster_offset = 0
    for l4_label, grp_idx in df_valid.groupby("ASSIGNED_L4_LABEL").groups.items():
        grp_X = X[grp_idx]
        k = min(N_CLUSTERS, len(grp_idx) // 10, len(grp_idx))
        if k < 2:
            df_valid.loc[grp_idx, "CLUSTER"] = cluster_offset
            cluster_offset += 1
            continue
        km = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=3, batch_size=2048)
        labels = km.fit_predict(grp_X)
        df_valid.loc[grp_idx, "CLUSTER"] = labels + cluster_offset
        cluster_offset += k
        print(f"  {l4_label}: {k} clusters")
else:
    # Full residual together
    print(f"Clustering full residual into {N_CLUSTERS} clusters ...")
    km = MiniBatchKMeans(
        n_clusters=N_CLUSTERS,
        random_state=RANDOM_STATE,
        n_init=5,
        batch_size=4096,
        verbose=0,
    )
    df_valid["CLUSTER"] = km.fit_predict(X)

cluster_sizes = df_valid["CLUSTER"].value_counts().sort_index()
print(f"\nCluster sizes (min={cluster_sizes.min():,}, max={cluster_sizes.max():,}, median={cluster_sizes.median():,.0f}):")
print(cluster_sizes.to_string())

## 6. Sample Listings per Cluster

For each cluster: show the dominant assigned-L4 label and sample product names.
Look for clusters where the dominant L4 looks **wrong** or **absent from the current anchor set** — those are candidates for a new L4.

In [ ]:
name_col = "PRODUCT_NAME" if "PRODUCT_NAME" in df_valid.columns else "PRODUCT_TEXT"

for cluster_id in sorted(df_valid["CLUSTER"].unique()):
    grp = df_valid[df_valid["CLUSTER"] == cluster_id]
    dominant_l4 = grp["ASSIGNED_L4_LABEL"].value_counts().idxmax()
    dominant_pct = grp["ASSIGNED_L4_LABEL"].value_counts().max() / len(grp) * 100
    avg_conf = grp["CONFIDENCE"].mean()

    print(f"\n{'='*70}")
    print(f"CLUSTER {cluster_id:3d} | {len(grp):6,} listings | dominant L4: {dominant_l4} ({dominant_pct:.0f}%) | avg confidence: {avg_conf:.3f}")
    print(f"{'='*70}")

    sample = grp.sample(min(SAMPLE_PER_CLUSTER, len(grp)), random_state=RANDOM_STATE)
    for _, row in sample.iterrows():
        name = str(row.get(name_col, ""))[:90]
        print(f"  [{row['CONFIDENCE']:.3f}] {name}")

## 7. (Optional) LLM Cluster Labeling

Only runs if `RUN_LLM_LABELING = True` in the config cell and Claude access is available.

In [ ]:
cluster_labels = {}

if RUN_LLM_LABELING:
    import json
    import boto3

    bedrock = get_bedrock_client(profile_name=AWS_PROFILE, region="us-east-1")
    clusters_to_label = sorted(df_valid["CLUSTER"].unique())[:LLM_MAX_CLUSTERS]

    for cluster_id in clusters_to_label:
        grp = df_valid[df_valid["CLUSTER"] == cluster_id]
        samples = grp.sample(min(20, len(grp)), random_state=RANDOM_STATE)[name_col].tolist()
        samples_text = "\n".join(f"- {s[:100]}" for s in samples)

        prompt = (
            "You are a product taxonomy expert for a life science and lab supply marketplace.\n"
            "Below is a sample of product names from one cluster. Provide:\n"
            "1. A short label (3–6 words) for this cluster\n"
            "2. One sentence explaining what unifies these products\n"
            "3. Whether this fits an existing L4 category or might need a NEW one\n"
            "Existing L4s: Chemicals & Solvents, Antibodies, Proteins & Peptides, "
            "Equipment Instruments & Parts, Lab Supplies & Consumables, Molecular Biology Reagents, "
            "Kits & Assays, Cell & Tissue Culture, Biospecimens, Animal Models, "
            "Controls Calibrators & Standards, Furniture & Storage, General Office Supplies.\n\n"
            f"Products:\n{samples_text}\n\n"
            "Respond in JSON: {\"label\": \"...\", \"description\": \"...\", \"fits_existing_l4\": true/false, \"suggested_l4\": \"...\"}"
        )

        try:
            response = bedrock.invoke_model(
                modelId=LLM_MODEL_ID,
                body=json.dumps({
                    "anthropic_version": "bedrock-2023-05-31",
                    "max_tokens": 300,
                    "temperature": LLM_TEMPERATURE,
                    "messages": [{"role": "user", "content": prompt}],
                }),
                contentType="application/json",
                accept="application/json",
            )
            raw = json.loads(response["body"].read())["content"][0]["text"]
            parsed = json.loads(raw)
            cluster_labels[cluster_id] = parsed
            flag = "** NEW L4? **" if not parsed.get("fits_existing_l4", True) else ""
            print(f"Cluster {cluster_id:3d}: {parsed.get('label')} {flag}")
        except Exception as e:
            print(f"Cluster {cluster_id:3d}: LLM error — {e}")
            cluster_labels[cluster_id] = {"label": f"cluster_{cluster_id}", "error": str(e)}
else:
    print("LLM labeling skipped (RUN_LLM_LABELING=False).")
    print("Review the sample printouts in Cell 6 and manually assign labels below.")

## 8. Manual Label Assignment

After reviewing Cell 6 output, fill in the dictionary below.
Mark `new_l4=True` for any cluster that doesn't fit the existing 13 anchors.

In [ ]:
# Fill this in after reviewing cluster samples above.
# Format: cluster_id -> {"label": "...", "suggested_l4": "...", "new_l4": True/False}
manual_labels = {
    # 0: {"label": "Specialty Solvents",     "suggested_l4": "Chemicals & Solvents", "new_l4": False},
    # 1: {"label": "Surgical Instruments",   "suggested_l4": "NEW",                  "new_l4": True},
}

if manual_labels:
    df_valid["CLUSTER_LABEL"] = df_valid["CLUSTER"].map(
        {k: v["label"] for k, v in manual_labels.items()}
    )
    new_l4_candidates = {k: v for k, v in manual_labels.items() if v.get("new_l4")}
    if new_l4_candidates:
        print("*** POTENTIAL NEW L4 CATEGORIES FOUND ***")
        for cid, info in new_l4_candidates.items():
            size = (df_valid["CLUSTER"] == cid).sum()
            print(f"  Cluster {cid} ({size:,} listings): {info['label']}")
    else:
        print("All clusters map to existing L4s — anchor descriptions may need minor tweaks but no new L4 required.")
else:
    print("Fill in manual_labels dict after reviewing cluster samples.")

## 9. Save Cluster Assignments

In [ ]:
out_cols = ["PRODUCT_ID", "ASSIGNED_L4_LABEL", "ASSIGNED_L4_ID", "CONFIDENCE", "CLUSTER"]
if "CLUSTER_LABEL" in df_valid.columns:
    out_cols.append("CLUSTER_LABEL")
if "PRODUCT_NAME" in df_valid.columns:
    out_cols.insert(1, "PRODUCT_NAME")

out_path = OUTPUT_DIR / f"residual_clusters_k{N_CLUSTERS}.csv"
df_valid[out_cols].to_csv(out_path, index=False)
print(f"Saved: {out_path} ({len(df_valid):,} rows)")

# Cluster summary
summary = df_valid.groupby("CLUSTER").agg(
    size=("PRODUCT_ID", "count"),
    avg_confidence=("CONFIDENCE", "mean"),
    dominant_l4=("ASSIGNED_L4_LABEL", lambda x: x.value_counts().idxmax()),
).sort_values("size", ascending=False)

if manual_labels:
    summary["manual_label"] = summary.index.map({k: v["label"] for k, v in manual_labels.items()})
    summary["new_l4"]       = summary.index.map({k: v.get("new_l4", False) for k, v in manual_labels.items()})

summary_path = OUTPUT_DIR / f"cluster_summary_k{N_CLUSTERS}.csv"
summary.to_csv(summary_path)
print(f"Summary: {summary_path}")
print()
print(summary.to_string())